# Fitted ECD-Threshold Comparison (Analysis A2)

## Protocol

**Status:** fixed on 2026-10-16, before any model in this notebook was run. Follow-up to analysis A (`10_beyond_ecd_comparison.ipynb`), recorded in git before execution.

### 1. Framing and disclosure

This is a **follow-up analysis**. It is not independent confirmation, and it is motivated by results already seen:

- Notebook 10 (pre-specified) found that the tabular and hybrid models significantly outperformed an **ECD-only Random Forest** (macro F1 +0.048 and +0.081, Holm p = 0.030 and 0.006).
- In the same notebook, a fixed two-threshold ECD rule scored 0.797 against 0.707 for the ECD-only Random Forest (descriptive). This suggested that the Random Forest was a weak comparator for "ECD alone". The rule's thresholds (2250/2500) were inferred from 2023 records after inspection.
- The baseline in this notebook (a depth-2 tree, i.e. at most three cut points) was chosen **knowing** that a two-threshold structure performs well on these tissues.

This analysis uses the same tissues, folds and models as notebooks 09–10. Its multiplicity correction covers only the two comparisons below, not notebooks 09–11 jointly.

### 2. Research question

Do the tabular and hybrid models improve on an ECD-only threshold model whose cut points are fitted within each training fold, when all are evaluated on the same unseen donors?

### 3. Cohort

Identical to notebook 10; the notebook asserts these counts and stops if any differ.

| Set | Expected |
|---|---|
| Primary cohort (non-2023, image-matched, year from `Tissue_ID`) | 551 tissues, 312 donors |
| **Evaluation set** (primary cohort with ECD) | **470 tissues, 280 donors**; A+ 244, A 159, B 67; years 2020: 78, 2021: 11, 2022: 54, 2024: 155, 2025: 172 |

### 4. Models

**ECD-only tree (new baseline):**

`DecisionTreeClassifier(max_depth=2, min_samples_leaf=10, class_weight="balanced", criterion="gini", random_state=42)`, with `ECD_CD_cells_per_mm2` as the only input.

- Trained on training-fold tissues with ECD.
- `max_depth=2` is the smallest depth that can express two thresholds separating three grades. The descriptive rule has exactly this form.
- `min_samples_leaf=10` prevents cut points defined by a handful of tissues.
- `class_weight="balanced"` matches the other models and the macro-averaged metric.
- These values are fixed here and are **not tuned**.

**Comparators (unchanged from notebook 10):**

| Model | Inputs | Training tissues |
|---|---|---|
| Tabular | 24 features, Random Forest as notebooks 07–10 | All training-fold tissues of the primary cohort |
| Hybrid | 24 features + frozen ResNet-50 → PCA 16, same Random Forest | All training-fold tissues of the primary cohort |
| ECD-only Random Forest (descriptive) | ECD alone, same Random Forest | Training-fold tissues with ECD |

**Reproducibility check:** the Tabular, Hybrid and ECD-only Random Forest fold-level scores must match `beyond_ecd_fold_scores.csv` from notebook 10, to within floating-point rounding (absolute difference < 1e-9). If they do not, the notebook stops and the discrepancy is recorded in the deviations log.

### 5. Validation

- Existing folds `Fold_r0` … `Fold_r4` from `donor_grouped_folds.csv`, restricted to the primary cohort. All fitting uses the training fold only.
- Metrics are computed on the evaluation-set tissues in each held-out fold.
- Checks before fitting:
  - no donor in both training and test;
  - every primary-cohort tissue is tested once per repeat;
  - every held-out fold has evaluation-set tissues of each grade.

### 6. Comparisons and metrics

| Role | Comparison | Metric |
|---|---|---|
| **Primary P1** | Tabular − ECD-only tree | Macro F1, per fold |
| **Primary P2** | Hybrid − ECD-only tree | Macro F1, per fold |
| Secondary | P1 and P2 | Balanced accuracy, accuracy, grade B recall (unadjusted) |
| Descriptive | ECD-only tree vs ECD-only Random Forest and vs the 2250/2500 rule | Macro F1 |
| Descriptive | All models | Per-grade precision and recall, pooled over repeats |
| Descriptive | ECD-only tree | Cut points learned in each of the 25 training folds, with their range |

### 7. Statistical analysis

- For P1 and P2: mean paired fold-level difference, corrected 95% CI and corrected repeated-CV t-test (Nadeau & Bengio, 2003; Bouckaert & Frank, 2004), 24 degrees of freedom.
  - The variance correction uses n_test / n_train from mean primary-cohort fold sizes.
- **Multiplicity:** Holm correction across P1 and P2. Confidence intervals are not multiplicity-adjusted.
- **Per-year (descriptive):** pooled difference by year and with each year excluded in turn; 2021 (11 tissues) flagged as too small.

### 8. Interpretation rules

Fixed before running and applied in code.

**Each primary comparison:**

| Result | Reported as |
|---|---|
| Difference > 0 and Holm-adjusted p < 0.05 | Model improved on the fitted ECD-only tree |
| Difference < 0 and Holm-adjusted p < 0.05 | Model performed worse than the fitted ECD-only tree |
| Otherwise | Inconclusive: no evidence of a difference from the fitted ECD-only tree |

**Answer to the research question:**

- *Evidence of improvement beyond a fitted ECD-only threshold model (follow-up analysis)*: at least one comparison is improved.
- *Improvement beyond ECD alone not established*: neither is improved and neither is worse.
- *Models performed worse than a fitted ECD-only threshold model*: either is worse. The specific model or models are named.

**Qualifiers:**

- **Grade B recall:** for each comparison, if the mean fold-level grade B recall difference is below −0.02, the result is reported with the reduced B recall and that model is not recommended over the ECD-only tree. As before, −0.02 is a project-defined decision rule, not a clinically validated threshold.
- **Per-year:** if a positive pooled difference becomes zero or negative when any single year is excluded, the result is reported as dependent on that year.

**Wording for reporting:**

- In all cases, notebook 10's result (improvement over an ECD-only Random Forest) is reported alongside this one, not replaced by it.
- This analysis is described as a follow-up motivated by notebook 10, not as independent confirmation.

**Other rules:**

- **No changes after results:** no other depths, leaf sizes, learners, cut-point methods, cohorts or metrics. Any further analysis is labelled post hoc.
- **Errors:** a code error may be fixed and the notebook rerun only if the fix does not change a protocol choice. Every rerun is logged in the deviations log with its reason.

## Deviations Log

Record any rerun or departure from the protocol here, with date and reason, before reporting results.

| Date | Change | Reason |
|---|---|---|
| — | None | — |